In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_main_nphc (monthly)
For every active **NPHC** row in the control table this notebook:

1. **Copies** new zips from the source into `<zip_tgt_path>/<mid_date>/`
2. **Unzips** them into `<unzip_tgt_path>/<mid_date>/<zip_name>/`
3. **Ingests** the JSON files into the target table, one row per file, tagged with the zip it came from

Just run it. It checks the whole source folder and only handles zips that aren't in `raw/` yet. The first run loads everything, and after that it only picks up what's new. A month that got missed is still new, so it's loaded next time together with the current month.

If a zip fails to unzip or ingest, its raw copy is deleted so the next run tries it again. Loading a file again replaces its earlier row, so you never get duplicates.

In [0]:
run_id          = str(uuid.uuid4())
control_entries = get_active_control_entries("NPHC")
ctl_by_sno      = {c["s_no"]: c for c in control_entries}
log_records     = []

print(f"run_id = {run_id}")
print(f"{len(control_entries)} active NPHC control row(s)")

### Preview: Available Files / New Files to load

In [0]:
this_month = datetime.now().strftime("%Y%m")
for ctl in control_entries:
    all_zips = list_zip_files(ctl["zip_source_path"])
    zips     = select_zips_to_copy(all_zips, ctl)
    print(f"s_no={ctl['s_no']}: {len(zips)} new zip(s); months (end date) to load: {nphc_months(zips) or '-'}")
    if this_month not in nphc_months(all_zips):
        print(f"  WARNING: no zip for the current month {this_month} at the source yet")

### Steps 1 and 2: copy and unzip

In [0]:
copy_and_unzip(run_id, control_entries, log_records)

### Step 3: ingest the JSON files

In [0]:
ingest_summary = {"SUCCESS": 0, "FAILED": 0}

unzipped = [r for r in log_records if r["stage"] == "UNZIP" and r["status"] == "SUCCESS"]
print(f"{len(unzipped)} unzipped zip file(s) to ingest")

for unzip_rec in unzipped:
    print(f"--- {unzip_rec['file_name']} ---")
    counts = nphc_ingest_zip(unzip_rec, ctl_by_sno[unzip_rec["s_no"]], run_id)
    for k, v in counts.items():
        ingest_summary[k] += v
    if counts["FAILED"] > 0:
        discard_raw_copy(unzip_rec["source_path"])      # so the next run tries this zip again

print("\nJSON ingestion summary:", ingest_summary)

### Run summary

In [0]:
display(spark.table(log_table).filter(F.col("run_id") == run_id).orderBy("start_time"))

In [0]:
display(spark.table(load_log_table).filter(F.col("run_id") == run_id).orderBy("started_at"))

In [0]:
# if anything failed, fail the job too (the details are in the log tables under this run_id)
failed = sum(1 for r in log_records if r["status"] == "FAILED") + ingest_summary["FAILED"]
if failed > 0:
    raise Exception(f"{failed} step(s) failed. See {log_table} / {load_log_table} where run_id = '{run_id}'")

In [0]:
%sql
select * from iics.casemix.nphc_json_test
order by file_id asc